In [14]:
from acspy import acsc
from acspy.control import Controller
import time
import sys
import numpy as np
import subprocess as sp
#This document has all the functions related to stage movement.

#We want to define a jog movement. This is a relative move, which moves a set distance in a given axes.
#The axis is an integer value from 0-2 (0 = x axis, 1 = y axis, 2 = z axis)
def jog(axis, distance, controller, boundary):
    
    #First off we check if the axis is valid. Since we have a XYZ stage, it should be an integer between 0-2
    if axis not in (0,1,2):
        print('Not a valid axis!')
        return 
    
    #We define some variables to use later.
    ax = controller.axes[axis]
    pos = ax.rpos
    
    #We check if the movement exceeds the boundary threshold. If it does it's important that the movement doesn't start
    if not (boundary[axis,0] <= pos+distance <= boundary[axis,1]):
        print('Movement exceeds boundary!')
        return
    
    #We enable the axis if it isn't enabled already.
    if ax.enabled==False:
        ax.enable()
        #A timer is applied here, as there is a short delay from enabling the stage to the stage being able to move
        time.sleep(0.5)
    
    #ax.ptpr is a relative move (jog). We send a signal to move the distance given in the function. 
    ax.ptpr(distance)
    #Now we want to check if the movement has stopped, and when it has stopped we want to inform the position and exit the function
    while True:
        if ax.in_position:
            break
    print('Movement has finished. from position,',pos,'to',ax.rpos)
    return



#We next want to define a multi axis movement. The input is a 1 dimensional array of 3 coordinates. 
#This function then checks if the movement is possible, and afterwards move in all axes simultanously.
def move_to(coordinates, controller, boundary):
    
    #First we want to check if the input is in the correct format. If it isn't the function stops to prevent errors.
    if len(coordinates) != 3:
        print('Coordinates are not in correct format! All 3 coordinates must be defined')
        return
    
    #We define the indidual controller axes
    ax_x = controller.axes[0]
    ax_y = controller.axes[1]
    ax_z = controller.axes[2]
    #We also define the current position for later control
    pos = [ax_x.rpos,ax_y.rpos,ax_z.rpos]
    
    #We enable all axes if any of them aren't enabled. Because of the potential delay, we won't enable if they're already enabled
    if False in [ax_x.enabled, ax_y.enabled, ax_z.enabled]:
        controller.enable_all()
        time.sleep(0.5)
        
    #We define a TRUE FALSE condition array. This array tests if each coordinate is within the boundary limits.
    boundary_check = [(boundary[0,0] <= coordinates[0] <= boundary[0,1]),
                     (boundary[1,0] <= coordinates[1] <= boundary[1,1]),
                     (boundary[2,0] <= coordinates[2] <= boundary[2,1])]
    
    #We know check if some of the coordinates are outside the boundary. If they are we inform which axes are exceeding. 
    #If any of them are exceeding the boundary, we stop the function to prevent damage to the equipment
    if False in boundary_check:
        if False in [boundary_check[0]]:
            print('X movement exceeds boundary')
        if False in [boundary_check[1]]:
            print('Y movement exceeds boundary')
        if False in [boundary_check[2]]:
            print('Z movement exceeds boundary')
        return
    
    #ptp is an absolute move to a specific coordinate. Even though the signal is sent sequentially, it's almost a simultanous movement
    ax_x.ptp(coordinates[0])
    ax_y.ptp(coordinates[1])
    ax_z.ptp(coordinates[2])
    
    #Last part we want to check if the movement has succeeded. If it has, inform the start and end position for control
    while True:
        if (ax_x.in_position == True) and (ax_y.in_position == True) and (ax_z.in_position == True):
            break
    print('Movement has finished. from position,',pos,'to',[ax_x.rpos, ax_y.rpos, ax_z.rpos])
    return


#This part of the code is just a test of the above. None of this is significant to the functions.
#First we define the controller and connect to it
controller = Controller(contype='ethernet',n_axes=3)
controller.connect()

#An important thing ALWAYS DEFINE BOUNDARIES
boundary = np.array([[-40, -20], [7, 14], [0, 4]])

#We use a smalle jog, to make sure it doesn't exceed limits even if the code fails
distance_x = 5

#Lastly we test the functions
jog(0,distance_x,controller,boundary)

#We disable the axes afterwards to make sure the functions work with disabled axes
controller.axes[0].disable()
if controller.axes[0].enabled==False:
    print('The X axis has sucesfully been disabled')
else:
    print('The X axis failed to disable')

#We do the same for the Y axis as was done with the X axis
distance_y = 2
jog(1,distance_y,controller,boundary)

controller.axes[1].disable()
if controller.axes[1].enabled==False:
    print('The Y axis has sucesfully been disabled')
else:
    print('The Y axis failed to disable')

    
#We do the same for the Z axis as was done with the X axis
distance_z = 1
jog(2,distance_z,controller,boundary)

controller.axes[2].disable()
if controller.axes[2].enabled==False:
    print('The Z axis has sucesfully been disabled')
else:
    print('The Z axis failed to disable')

#Now we want to test the move_to function. The setup of boundaries and connecting to the controller is already done.
#We now define the coordinates we want to move to. 
coordinates =[-30,10,0]

#We then use the function using the coordinates, controller and previously defined boundary
move_to(coordinates, controller, boundary)

#Now we want to make sure each axes is disabled. 
controller.disable_all()
if controller.axes[0].enabled==False and controller.axes[1].enabled==False and controller.axes[2].enabled==False:
    print('All axes has sucesfully been disabled')
else:
    if controller.axes[0].enabled==True:
        print('The X axis failed to disable')
    if controller.axes[1].enabled==True:
        print('The Y axis failed to disable')
    if controller.axes[2].enabled==True:
        print('The Z axis failed to disable')

#Lastly ALWAYS DISCONNECT THE CONTROLLER. 
#Failure to do so can lead to the port clogging up and being unable to send signals to it.
#If the port is clogged up, open "SPIIplus user mode driver" and disconnect each "connected application"
controller.disconnect()


Movement has finished. from position, -29.94064453125 to -24.94064453125
The X axis has sucesfully been disabled
Movement has finished. from position, 10.0126171875 to 12.012753906250001
The Y axis has sucesfully been disabled
Movement has finished. from position, -0.00017500000000000003 to 0.999825
The Z axis has sucesfully been disabled
Movement has finished. from position, [-25.1986328125, 12.005273437500001, 0.9998100000000001] to [-30.0, 10.0, 0.0]
All axes has sucesfully been disabled


np.False_